In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

RAW = Path("data/raw")
OUT = Path("data/processed")
OUT.mkdir(parents=True, exist_ok=True)

In [2]:
SERIES = {
    "INDPRO":       {"freq": "M", "tf": "log_diff", "label": "Industrial Production"},
    "RSAFS":        {"freq": "M", "tf": "log_diff", "label": "Retail Sales"},
    "PAYEMS":       {"freq": "M", "tf": "log_diff", "label": "Nonfarm Payrolls"},
    "UNRATE":       {"freq": "M", "tf": "diff",     "label": "Unemployment Rate"},
    "HOUST":        {"freq": "M", "tf": "log_diff", "label": "Housing Starts"},
    "PCE":          {"freq": "M", "tf": "log_diff", "label": "Personal Consumption"},
    "DGORDER":      {"freq": "M", "tf": "log_diff", "label": "Durable Goods Orders"},
    "ICSA":         {"freq": "W", "tf": "log_diff", "label": "Initial Claims"},
    "T10Y2Y":       {"freq": "D", "tf": "level",    "label": "10Y-2Y Spread"},
}
TARGET = "A191RL1Q225SBEA"

In [3]:
def load_series(code):
    df = pd.read_csv(RAW / f"{code}.csv", index_col=0, parse_dates=True)
    s = df.iloc[:, 0]
    s.name = code
    return s

def transform(s, rule):
    if rule == "log_diff":
        return np.log(s.replace(0, np.nan)).diff()
    if rule == "diff":
        return s.diff()
    return s

cols = {}
for code, information in SERIES.items():
    s = load_series(code)
    s = transform(s, information["tf"])
    s = s.resample("ME").mean().resample("QE").mean()
    cols[information["label"]] = s

gdp = load_series(TARGET)
gdp.index = gdp.index.to_period("Q").to_timestamp("Q")
cols["GDP Growth"] = gdp

panel = pd.DataFrame(cols).dropna(how="all")
panel.shape

(146, 10)

In [4]:
summary = panel.describe().T
summary = summary[["count", "mean", "min", "max", "std"]]
summary.columns = ["N", "Mean", "Min", "Max", "Std Dev"]
summary.round(5)

,N,Mean,Min,Max,Std Dev
Industrial Production,145.0,0.00117,-0.02148,0.01555,0.00505
Retail Sales,137.0,0.00374,-0.03340,0.03800,0.00715
Nonfarm Payrolls,145.0,0.00086,-0.03057,0.00978,0.00310
Unemployment Rate,145.0,-0.00345,-1.06667,2.20000,0.23236
Housing Starts,145.0,0.00001,-0.12712,0.07577,0.03232
Personal Consumption,145.0,0.00406,-0.02244,0.01939,0.00355
Durable Goods Orders,137.0,0.00268,-0.07120,0.04682,0.01708
Initial Claims,146.0,-0.00015,-0.10544,0.27028,0.02585
10Y-2Y Spread,146.0,1.00310,-0.76864,2.80237,0.90557
GDP Growth,144.0,2.57917,-28.00000,34.90000,4.42420
